# LS-NEQ 02_switching_grid — Sprachwechsel-Raster und Counterfactual-Attribution

**Dossier-Stufe C** (`lsne.experiments.STAGES["C"]`, siehe `docs/DESIGN.md` §8).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Dossier-Kernkontraste auf dem Raster K × C: Texteffekt T(K, C) = ln p₀(gemischt) − ln p₀(K=0), Interventionsnutzen B(x) = ln p_δ − ln p₀ als Dosiskurve pro Richtung (ent, lang_dom, freq, Zufallsrichtungen, jeweils ±) und Wechselwirkung I = B(gemischt) − B(K=0); dazu die Counterfactual-Attribution nach Im et al.: Patching einzelner Tokenklassen (Whitespace, Satzzeichen, Grenz-Tokens, …) des gemeinsamen englischen Schlussabschnitts von x_mono in x_mixed mit Zufallspositions-Null, sowie Gradient × Aktivierung.

**Eingaben.** Stimulusbank, Richtungsbank (aus dem Stufe-B-Lauf auf Drive, sonst neu gebaut).

**Ausgaben.** Tabellen `grid`, `grid_summary`, `contrasts`, `contrasts_boot`, `attribution`, `attribution_summary`, `grad_x_act`, `grad_x_act_by_class`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/C/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/02_switching_grid/`; am Ende eine kurze `summary_02_switching_grid_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Modell → Stufe C → T(K,C) mit Familien-Bootstrap → B(x)-Dosiskurven → Attribution nach Tokenklasse vs. Zufall → Zusammenfassung.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
import os

try:
    from google.colab import drive            # type: ignore
    drive.mount("/content/drive")
except Exception as e:                        # not on Colab, or the mount was refused
    print("Kein Colab-Drive:", repr(e).split("(")[0], "-> lokale Ausgabe")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "02_switching_grid"
STAGE = "C"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(status="reused", run_dir=str(rd.path), seconds=0.0, error="")
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(status="ok", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="")
        print(f"Stufe {stage} ok in {report[stage]['seconds']:.0f} s: {', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Stufe C ausführen ----------------------------------------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage C --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT>
res_C = run_stage("C")
print({k: len(v) for k, v in res_C.items()})


In [ ]:
# --- T(K, C): Texteffekt mit Familien-Cluster-Bootstrap -------------------------------
# Families are the statistical unit (DESIGN §9): resample whole families with replacement.
with guarded("T(K, C) mit Familien-Bootstrap"):
    from lsne.stats import cluster_bootstrap

    con = table("C", "contrasts")
    std = con[con["kind"] == "standard"].drop_duplicates("tag")     # T does not depend on direction / dose
    rows = []
    for (K, Cc), g in std.groupby(["K", "C"], sort=True):
        r = cluster_bootstrap(g, "T", "family", n_boot=limits.n_boot, seed=SEED)
        rows.append(dict(K=int(K), C=Cc, T=r["mean"], ci_low=r["ci_low"], ci_high=r["ci_high"],
                         n_families=r["n_clusters"], n_variants=r["n_rows"]))
    tb = pd.DataFrame(rows)
    show(tb)
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(tb))
    colors = {"C1": "C0", "C2": "C1", "C3": "C2", "seg": "C7"}
    ax.bar(x, tb["T"], yerr=[(tb["T"] - tb["ci_low"]).clip(lower=0), (tb["ci_high"] - tb["T"]).clip(lower=0)],
           capsize=4, color=[colors.get(c, "C4") for c in tb["C"]])
    ax.set_xticks(x)
    ax.set_xticklabels([f"K={k}\n{c}" for k, c in zip(tb["K"], tb["C"])])
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylabel("T = ln p₀(gemischt) − ln p₀(K=0)  [nat]")
    ax.set_title(f"Texteffekt T(K, C), 95 %-CI aus {limits.n_boot} Familien-Bootstraps")
    savefig("T_by_K_C")
    plt.show()
    boot = table("C", "contrasts_boot")
    if len(boot):
        print("Vorab berechnete Kontraste (Stufe C, inkl. I pro Richtung/Dosis):")
        show(boot[boot["quantity"] == "T"])


In [ ]:
# --- B(x): Dosiskurven pro Richtung (Mittel über Varianten, Linien pro K) --------------
with guarded("B(x) Dosiskurven"):
    gs = table("C", "grid_summary")
    gs = gs[gs["kind"] == "standard"] if "kind" in gs else gs
    fams = [f for f in ["ent", "lang_dom", "freq", "rand"] if f in set(gs["dir_family"])]
    fig, axes = plt.subplots(1, len(fams), figsize=(4.3 * len(fams), 3.9), sharey=True, squeeze=False)
    for ax, fam in zip(axes[0], fams):
        sub = gs[gs["dir_family"] == fam]
        for i, (K, g) in enumerate(sorted(sub.groupby("K"))):
            for sign, ls in ((1, "-"), (-1, "--")):
                m = g[g["sign"] == sign].groupby("eps")["B"].mean()
                ax.plot(m.index, m.values, ls, color=f"C{i}", marker="o", ms=3,
                        label=f"K={K} ({'+' if sign > 0 else '−'})")
        ax.set_xscale("symlog", linthresh=1e-3)
        ax.axhline(0, color="k", lw=0.8)
        ax.set_title(f"Richtung {fam}" + (" (Mittel über Zufallsrichtungen)" if fam == "rand" else ""), fontsize=9)
        ax.set_xlabel("Dosis ε (relativ zur RMS des Residuums)")
    axes[0, 0].set_ylabel("B = ln p_δ(Ziel) − ln p₀(Ziel)")
    axes[0, -1].legend(fontsize=7, ncol=2)
    savefig("B_dose_curves")
    plt.show()
    # interaction I = B(mixed) - B(K=0) at the dossier dose 0.01
    con = table("C", "contrasts")
    sub = con[(con["kind"] == "standard") & (con["eps"] == 0.01) & (con["sign"] == 1)]
    if len(sub):
        print("I = B(gemischt) − B(K=0) bei ε = 0.01, Vorzeichen +, Mittel über Familien:")
        show(sub.groupby(["dir_family", "K", "C"])["I"].agg(["mean", "std", "count"]))


In [ ]:
# --- Counterfactual-Attribution: Wiederherstellung R nach Tokenklasse vs. Zufall ---------
# R_c = (ln p_patched - ln p_mixed) / (ln p_mono - ln p_mixed); random control = the same
# number of random tail positions (Im et al. 2026: delimiters carry a disproportionate share).
with guarded("Attribution nach Tokenklasse"):
    att = table("C", "attribution_summary")
    for pair, g in att.groupby("pair"):
        layers = sorted(g["layer"].unique())
        fig, axes = plt.subplots(1, len(layers), figsize=(3.4 * len(layers), 3.8), sharey=True, squeeze=False)
        for ax, l in zip(axes[0], layers):
            s = g[g["layer"] == l].set_index("class").reindex(list(C.PATCH_CLASSES))
            x = np.arange(len(s))
            ax.bar(x - 0.2, s["recovery_mean"], 0.4, yerr=s["recovery_sd"].fillna(0), capsize=2, label="Tokenklasse")
            ax.bar(x + 0.2, s["recovery_random_mean"], 0.4, color="grey", label="Zufallspositionen (gleiche Anzahl)")
            ax.set_xticks(x)
            ax.set_xticklabels(s.index, rotation=70, fontsize=8)
            ax.axhline(0, color="k", lw=0.8)
            ax.axhline(1, color="k", lw=0.5, ls=":")
            ax.set_title(f"{pair}: Block-Eingang {l}", fontsize=9)
        axes[0, 0].set_ylabel("Wiederherstellung R")
        axes[0, 0].legend(fontsize=7)
        savefig(f"attribution_{pair}")
        plt.show()
    if len(att):
        print("Überschuss über Zufall (recovery_mean − recovery_random_mean):")
        show(att.pivot_table(index=["pair", "class"], columns="layer", values="excess_over_random"), n=60)

with guarded("Gradient × Aktivierung: Anteil pro Tokenklasse"):
    gx = table("C", "grad_x_act_by_class")
    std = gx[gx["kind"] == "standard"] if "kind" in gx else gx
    piv = std.groupby(["layer", "class"])["abs_share"].mean().unstack("class")
    ax = piv.plot(kind="bar", figsize=(9, 3.8), width=0.8)
    ax.set_ylabel("mittlerer Anteil an Σ|grad × act|")
    ax.set_xlabel("Block-Eingang")
    ax.set_title("Gradient × Aktivierung von ln p(Ziel) nach Tokenklasse")
    ax.legend(fontsize=7, ncol=4)
    savefig("grad_x_act_share")
    plt.show()


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
